# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [ ]:
## These are most likely the libraries you will use
# Add or remove imports as needed
# !pip install hvplot

import pandas as pd
import numpy as np
import hvplot.pandas

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Geospatial / geocoding
# Run this if geopy isn't already installed (can comment out after installed)
# !pip install geopy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# Panel
# Run if panel isn't already installed (can comment out after installed)
# !pip install panel
# !pip install jupyter_bokeh
import panel as pn
pn.extension('plotly')

In [ ]:
### Load your dataset here
df = pd.read_csv("cleaned_games_dataset.csv")

In [ ]:
### Check the head/info of the dataset
# df.head()
df.info()


## Question 1: Analytical Question
Write a question about your data that can be explored with an interactive plot. A good example would be a question where you can compare different categories. Explain how the plot help answer your question and why you choose that plot type.

Examples:
- Which regions have the highest average values?
- How does one variable compare across time?

In [ ]:
# Your code . . .
### Question
# Which game features are the strongest predictors of price?

### Why this question matters
# This question is useful because our group ultimately wants to build a machine learning model that
# predicts a proper game price. Before building a model, it is important to explore which features appear
# to have the strongest relationship with price. By interactively comparing price against features like
# recommendations, review scores, genre, and free-to-play status, we may be able to identify which variables
# would be the most useful predictors.

### Why this plot type works
# An interactive scatter plot is a good choice because it allows us to compare price against different
# numeric features one at a time while keeping the same visual structure. Using a dropdown menu makes the analysis
# more flexible, since users can switch between possible predictors and see whether the relationship with price
# looks strong, weak, or unclear. Coloring by free-to-play status and showing genre in the hover information would
# add context as well without cluttering the graph.

### How the plot helps answer the question
# If a feature shows a clearer trend or clustering pattern with price, it may be a stronger candidate for a future
# machine learning model. If the points appear very scattered with no visible pattern, that feature may be less
# useful for price prediction.

## Question 2. Create a simple interaction plot
Create a plot, it can be related to your question #1 or a new question, that users can interact with in some meaningful way. Explain in a markdown, how does the interaction add to the analysis.

Example of possible interactions:
*   Hover over information
*   Toogle between groups

In [ ]:
# Your code . . .
# Create a dropdown for selecting feature
feature_options = ["metacritic_score", "recommendation_total", "mat_initial_price"]

# Scatter plot with dropdown
fig = px.scatter(
    df,
    x="metacritic_score",
    y="mat_final_price",
    color="is_free",
    hover_data=["name"],
    title="Feature vs Final Price"
)

# Dropdown menu
fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(label="Metacritic Score",
                     method="update",
                     args=[{"x": [df["metacritic_score"]]},
                           {"xaxis": {"title": "Metacritic Score"}}]),

                dict(label="Recommendations",
                     method="update",
                     args=[{"x": [df["recommendations_total"]]},
                           {"xaxis": {"title": "Total Recommendations"}}]),

                dict(label="Initial Price",
                     method="update",
                     args=[{"x": [df["mat_initial_price"]]},
                           {"xaxis": {"title": "Initial Price"}}]),
            ],
            direction="down"
        )
    ]
)

fig.show()

The dropdown interaction allows users to switch between different potential predictors of game price while keeping the same visual structure. This makes it easier to compare relationships without generating multiple separate plots.

For example, users can quickly observe whether:
*  Higher Metacritic scores are associated with higher prices
*  More recommended games tend to cost more
* Initial price strongly influences final price

More recommended games tend to cost more
Initial price strongly influences final price

This interaction improves efficiency and clarity by allowing direct comparison across variables in a single visualization, making patterns and trends easier to identify.

## Question 3. Choropleth Planning
Design a choropleth idea using your dataset.
In a markdown cell:
*  Identify the geographic unit you would map (state, county, country, ZIP code, etc.)
*  Identify the variable you would color by
*  Explain if any aggregation or preprocessing is needed
*  Briefly describe what GeoJSON file would be required

You do not need to have a perfect dataset for this question. However, your plan should be realistic.

If your data does not fully support a choropleth, build a prototype table that explains that structure you would need before mapping.

Unfortunately our dataset does not contain any geographical information. Therefore, it does not directly support creating a choropleth map. But we can use our data to create a potential tactic of creating columns so it does support a chloropleth.

For the geographic unit we could map, we could use country of origin for a game and where they are most popular. For instance, a game like CS2 would be most popular somewhere like the US but would be less popular in another country. Laying out where each game is more popular.

The variable we could color by would be recommendations_total because we are looking at where a game is most popular. This entails looking at geographic data of popularity by country and where they are recommended the most.

We would definitely need a combination of aggregation and preprocessing in order to get it working with a chloropleth. We would first need to add the geographic column which details every game with their most popular country. That way we can categorize the data properly. Once that is done, we can group the data togetehr and find the mean so that we can have a mix with the popular country and the average recommendations per country for a game. We will handle missing data by dropping it. The table structure would look like what is done below.

The GeoJSON file that is needed is one that contains boundaries for each geographic unit like what country is popular with what. The labels that are used from the dataset need to match the ones used in the GeoJSON.


In [ ]:
# Country      | Avg_Recommendations
# ----------------------------------
# USA          | 15234
# UK           | 12890
# Canada       | 9345

## Question 4. Geospatial Possibility Check

Determine whether your dataset can support a map-based visualization.

In a markdown cell, answer one of the following:
- If **yes**, explain what geographic fields you have and what type of map is appropriate.
- If **no**, explain what is missing and what you would need to create a map.

Write code that inspects or prepares the geographic column(s) you may use.

In [ ]:
# Your code . . .
print("Columns in dataset:")
print(df.columns)

Our dataset does not support a map-based visualization because it lacks geographic fields such as country, state, city, or coordinates (latitude/longitude) in which you can see above there are no columns matching that.

What is missing:
* No location-based columns
* No geographic identifiers to map data points

What would be needed to create a map visualization:

* A country or region column indicating where a game is most popular
* Developer or publisher location
* User distribution data by region
* Latitude and longitude coordinates

## Question 5. Geopy / Location Preparation

If your dataset has location names, addresses, cities, states, or countries, use geopy on a sample of your data to geocode locations or validate location information.

If your dataset does not have data that contains location, pick 5 destination you want to visit and use geopy to geocode locations.

In [ ]:
# Your code . . .
### Method was taken from M06_Colab16
## Function to find the coordinate of a given city
def findGeocode(city):
    ''' try-except is used to overcome the exception thrown by geolocator
    using geocodertimedout '''
    try:
        # Specify the user_agent as your app name and it should not be none
        geolocator = Nominatim(user_agent="CS133")
        return geolocator.geocode(city)
    except GeocoderTimedOut:
        return None

## Pick 5 locations to geocode
locations = [
    "Tokyo, Japan",
    "Paris, France",
    "New York, USA",
    "Sydney, Australia",
    "Rome, Italy"
]

## Loop through the locations and find their coordinates
results = []

for place in locations:
    location = findGeocode(place)
    
    results.append({
        "Location": place,
        "Latitude": location.latitude,
        "Longitude": location.longitude
    })
    
results

## Question 6. Panel Widget

Create a Panel Widget that controls something in your analysis such as the ability to choose a column, category, year, etc.

The widget should affect an output such as a plot, table, or summary statistic.

In [ ]:
pn.extension()
pn.extension('tabulator')

### Copy dataframe
q6_df = df.copy()

### Use final price as target
price_col = "mat_final_price"

### Select which columns would be relevant in predicting game price
numeric_cols = q6_df.select_dtypes(include='number').columns.tolist()
genre_cols = ["Action","Adventure","Casual","Free to Play","Indie",
              "Massively Multiplayer","Other","RPG","Racing",
              "Simulation","Sports","Strategy","Unknown", "has_metacritic",
              "mat_initial_price", "mat_discount_percent"]

feature_list = [
    col for col in numeric_cols
    if col != price_col and col not in genre_cols
]

### Interactive dataframe (like class)
idf = q6_df.interactive()

## Dropdown widget
feature_select = pn.widgets.Select(
    name='Select Feature',
    options=feature_list,
    value=feature_list[0]
)

## Interactive scatter plot
feature_plot = (
    idf[[price_col, feature_select.param.value]]
    .dropna()
    .hvplot.scatter(
        x=feature_select.param.value,
        y=price_col,
        width=700,
        height=400,
        title='Feature vs Final Price'
    )
)

## Layout
q6_panel = pn.Column(
    "## Question 6: Interactive Panel Widget",
    "Explore how ratings affect final game pricing.",
    pn.Row(
        feature_plot
    )
)

q6_panel

## Question 7. Mini Dashboard

Build a small Panel dashboard with at least two components. Arrange the components so that it is in a readable layout. Your dashboard should include:
* At least one plot,
* An additional element of your choice such as a widget, table, second plot, etc.

In [ ]:
# Write your code here

import panel as pn
import hvplot.pandas

pn.extension()
pn.extension('tabulator')

### Copy dataframe
q7_df = df.copy()

### Use interactive dataframe
idf = q7_df.interactive()

### Price column
price_col = "mat_final_price"

### Exclude same columns as Q6
genre_cols = ["Action","Adventure","Casual","Free to Play","Indie",
              "Massively Multiplayer","Other","RPG","Racing",
              "Simulation","Sports","Strategy","Unknown", "has_metacritic",
              "mat_initial_price", "mat_discount_percent"]

### Numeric feature choices
numeric_cols = q7_df.select_dtypes(include='number').columns.tolist()
feature_list = [col for col in numeric_cols if col != price_col and col not in genre_cols]

### Widgets
select = pn.widgets.Select(
    name='Select Feature',
    options=feature_list,
    value=feature_list[0]
)

color_toggle = pn.widgets.Checkbox(
    name="Color by Free vs Paid",
    value=False
)

### Interactive table
q7_table = (
    idf[["name", price_col, select.param.value]]
    .dropna()
    .head(15)
)

### Plot function to make the color toggle reactive
def make_plot(feature, color_on):
    data = q7_df[[price_col, feature, "is_free"]].dropna()

    return data.hvplot.scatter(
        x=feature,
        y=price_col,
        by="is_free" if color_on else None,
        width=700,
        height=400,
        title="Feature vs Final Price"
    )

## Bind widgets to plot
q7_plot = pn.bind(make_plot, select, color_toggle)

### Mini dashboard layout
q7_dashboard = pn.Column(
    "## Question 7: Mini Dashboard",
    color_toggle,
    pn.Row(
        q7_table.hvplot.table(width=400),
        q7_plot
    )
)

q7_dashboard

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
- What was the hardest part of working with your dataset?
- Did implementing interactive tools help your analysis? Why or why not?
- If you had more time, what would you improve or add?

In [ ]:
# Write your code here


The most useful interactive element in our notebook was the feature-selection dropdown used in the Panel visualizations. It made it much easier to compare how different variables, such as metacritic score, recommendations, and initial price, related to the final game price without needing to build a separate graph for each feature. The free-versus-paid color toggle was also helpful because it added another layer of comparison and made the scatter plots more informative.

The hardest part of working with this dataset was making sure the pricing data was meaningful and correctly interpreted. Some columns were closely related to each other, such as `mat_initial_price` and `mat_final_price`, while other values appeared to contain extreme outliers. This made it harder to decide which variables were actually useful for analysis and which ones needed to be cleaned or excluded.

Implementing interactive tools did help our analysis because they made it easier to explore patterns quickly and compare multiple features in a more flexible way. Instead of generating many static plots, we could use one interface to test several possible relationships and better understand which variables might be useful for predicting game price.

If we had more time, we would improve the outputs by adding more filters, such as genre-based filtering or price-range selection. Currently, our graphs just give information on the entire dataset, and we would like to give the user the freedom to create their own target price ranges. 